# Comparison of Generative Data Sampling

Code is from [This Github Repo](https://github.com/Diyago/Tabular-data-generation/tree/master) to save time

basically will be running all four options and comparing outputs to show how these vary and if there is noticable difference for the quality of the data and other implications for synthetic data to improve dataset size (like if that causes bias?)


In [ ]:
import kagglehub
import pandas as pd
import ipaddress
from sklearn.preprocessing import LabelEncoder

# Download latest version to compare with the two generated datasets from models
path = kagglehub.dataset_download(
    "munaalhawawreh/xiiotid-iiot-intrusion-dataset")

print("Path to dataset files:", path)
print(path)
df = pd.read_csv(path + "\\X-IIoTID dataset.csv", skipinitialspace=True)


# 1. Basic Cleaning: Strip whitespace from columns and string values
df.columns = df.columns.str.strip()
df = df.apply(lambda x: x.str.strip() if x.dtype == "object" else x)

# 2. Universal IP & Port Normalizer (Handles IPv4, IPv6, ?, -, and Ports)


def universal_to_numeric(val):
    val = str(val).strip()
    if val in ['?', '-', '', 'None']:
        return 0
    try:
        # If it looks like an IP, convert to int
        if '.' in val or ':' in val:
            return int(ipaddress.ip_address(val))
        # Otherwise, treat as a standard number
        return int(float(val))
    except:
        return 0


for col in ['Scr_IP', 'Des_IP', 'Scr_port', 'Des_port']:
    df[col] = df[col].apply(universal_to_numeric)

# 3. Timestamp Fix (Convert Unix to Numeric safely)
df['Timestamp'] = pd.to_numeric(
    df['Timestamp'], errors='coerce').fillna(0).astype(int)
df['Time'] = pd.to_datetime(df['Timestamp'], unit='s')
df.drop(columns=['Timestamp'], inplace=True)
# 4. Boolean/Flag Mapping (TRUE/FALSE strings or actual Bools -> 0/1)
bool_cols = [
    'is_syn_only', 'Is_SYN_ACK', 'is_pure_ack', 'is_with_payload',
    'FIN or RST', 'Bad_checksum', 'is_SYN_with_RST', 'anomaly_alert'
]
for col in bool_cols:
    if col in df.columns:
        # Map strings if they exist, otherwise force to int
        if df[col].dtype == 'object':
            df[col] = df[col].str.upper().map(
                {'TRUE': 1, 'FALSE': 0, '-': 0, '?': 0})
        df[col] = df[col].fillna(0).astype(int)

# 5. Mass Numeric Conversion for Metrics (Duration, Bytes, CPU stats, etc.)
# We replace '?' and '-' with 0 first
df.replace(['?', '-'], 0, inplace=True)
cat_cols = ['Protocol', 'Service', 'class1', 'class2', 'class3']
encoders = {}
# 6. Categorical Cleanup
for col in ['Protocol', 'Service']:
    df[col] = df[col].replace(0, 'unknown').astype(str)
for col in cat_cols:
    if col in df.columns:
        le = LabelEncoder()
        # Ensure everything is a string before encoding to avoid mixed-type errors
        df[col] = le.fit_transform(df[col].astype(str))
        encoders[col] = le

# 3. Final safety check: ensure NO strings remain in the numeric features
# (This prevents the 'udp' error once and for all)
for col in df.columns:
    if df[col].dtype == 'object':
        # If any stray strings exist, force them to numeric or drop
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)
# Identify all columns that should be numeric (excluding categorical labels)

numeric_cols = [c for c in df.columns if c not in cat_cols + ['Date']]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

# 7. Final Drop
df.drop(columns=['Date'], inplace=True)

df.info()

c:\Users\antho\.conda\envs\CS9840\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Path to dataset files: C:\Users\antho\.cache\kagglehub\datasets\munaalhawawreh\xiiotid-iiot-intrusion-dataset\versions\1
C:\Users\antho\.cache\kagglehub\datasets\munaalhawawreh\xiiotid-iiot-intrusion-dataset\versions\1


C:\Users\antho\AppData\Local\Temp\ipykernel_13640\1224735001.py:12: DtypeWarning: Columns (0: Timestamp, 1: Scr_port, 2: Des_port, 3: missed_bytes, 4: anomaly_alert) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path + "\\X-IIoTID dataset.csv", skipinitialspace=True)


<class 'pandas.DataFrame'>
RangeIndex: 820834 entries, 0 to 820833
Data columns (total 67 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   Scr_IP                       820834 non-null  float64
 1   Scr_port                     820834 non-null  int64  
 2   Des_IP                       820834 non-null  float64
 3   Des_port                     820834 non-null  int64  
 4   Protocol                     820834 non-null  int64  
 5   Service                      820834 non-null  int64  
 6   Duration                     820834 non-null  float64
 7   Scr_bytes                    820834 non-null  int64  
 8   Des_bytes                    820834 non-null  int64  
 9   Conn_state                   820834 non-null  int64  
 10  missed_bytes                 820834 non-null  float64
 11  is_syn_only                  820834 non-null  int64  
 12  Is_SYN_ACK                   820834 non-null  int64  
 13  is_pure_ac

In [2]:
df.head()

,Date,Timestamp,Scr_IP,Scr_port,Des_IP,Des_port,Protocol,Service,Duration,Scr_bytes,...,OSSEC_alert_level,Login_attempt,Succesful_login,File_activity,Process_activity,read_write_physical.process,is_privileged,class1,class2,class3
0,9/01/2020,1578540956,192.168.2.199,49278,192.168.2.10,80,tcp,http,0.67369,13437,...,5,0,0,0,0,0,0,Scanning_vulnerability,Reconnaissance,Attack
1,13/01/2020,1578871873,10.0.1.5,39769,131.236.3.92,53,udp,dns,0.000083,78,...,0,0,0,0,0,0,0,Normal,Normal,Normal
2,9/01/2020,1578522486,172.24.1.80,59050,172.24.1.1,53,udp,dns,0.000132,38,...,0,0,0,0,0,0,0,Normal,Normal,Normal
3,27/02/2020,1582757640,192.168.2.196,37966,192.168.2.10,1880,tcp,websocket,9.378481,1121,...,0,1,1,1,1,1,1,Normal,Normal,Normal
4,16/12/2019,1576452612,172.24.1.80,38233,172.24.1.1,53,udp,dns,0.000074,-,...,0,0,0,0,0,0,0,Normal,Normal,Normal


In [3]:
df.describe()

,Conn_state,OSSEC_alert,OSSEC_alert_level,Login_attempt,Succesful_login,File_activity,Process_activity,read_write_physical.process,is_privileged
count,820834.000000,820834.000000,820834.000000,820834.000000,820834.000000,820834.000000,820834.000000,820834.000000,820834.000000
mean,0.851623,0.051186,0.267717,0.087305,0.082730,0.072650,0.082620,0.355309,0.082456
std,0.355474,0.220377,1.178027,0.282282,0.275475,0.259562,0.275307,0.478607,0.275059
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000
max,1.000000,1.000000,10.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [4]:
from sklearn.model_selection import train_test_split
import plotly.graph_objects as go

target_cols = ['class1', 'class2', 'class3']

X = df.drop(columns=target_cols)
y = df[['class1']]  # Focus on class1 for now

# Create the 1,000 row set
X_train_small, X_test_small, y_train_small, y_test_small = train_test_split(
    X, y, train_size=800, test_size=200, stratify=y, random_state=42
)

# Create the 10,000 row set
X_train_large, X_test_large, y_train_large, y_test_large = train_test_split(
    X, y, train_size=8000, test_size=2000, stratify=y, random_state=42
)

original_dist = df['class1'].value_counts(normalize=True).sort_index()
small_test_dist = y_test_small['class1'].value_counts(
    normalize=True).sort_index()
large_test_dist = y_test_large['class1'].value_counts(
    normalize=True).sort_index()

# 2. Create the Figure
fig = go.Figure(data=[
    go.Bar(
        name='Original Dataset',
        x=original_dist.index,
        y=original_dist.values,
        text=[f'{p:.1%}' for p in original_dist.values],
        textposition='auto',
        marker_color='#1f77b4'
    ),
    go.Bar(
        name='Small Test Set',
        x=small_test_dist.index,
        y=small_test_dist.values,
        text=[f'{p:.1%}' for p in small_test_dist.values],
        textposition='auto',
        marker_color='#ff7f0e'
    ),
    go.Bar(
        name='Large Test Set',
        x=large_test_dist.index,
        y=large_test_dist.values,
        text=[f'{p:.1%}' for p in large_test_dist.values],
        textposition='auto',
        marker_color="#1aff0e"
    )
])

fig.update_layout(
    # 'total ascending' makes the largest bar top-most in a horizontal plot
    xaxis={'categoryorder': 'total ascending'},
    showlegend=False,
    width=1000,
    height=800
)

fig.show()

pd.set_option('display.expand_frame_repr', False)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)  # Ensure the width is large enough

orig_counts = df['class1'].value_counts()
orig_pcts = df['class1'].value_counts(normalize=True) * 100

# 2. Calculate counts and percentages for the Test Set
small_test_counts = y_test_small['class1'].value_counts()
small_test_pcts = y_test_small['class1'].value_counts(normalize=True) * 100

large_test_counts = y_test_large['class1'].value_counts()
large_test_pcts = y_test_large['class1'].value_counts(normalize=True) * 100

# 3. Combine into a single detailed DataFrame
comparison_table = pd.DataFrame({
    'Original (count)': orig_counts,
    'Original (%)': orig_pcts,
    'Small Test Set (count)': small_test_counts,
    'Small Test Set (%)': small_test_pcts,
    'Large Test Set (count)': large_test_counts,
    'Large Test Set (%)': large_test_pcts
})

# 4. Optional: Add a column to show the percentage of the total that went to the test set
# (This should be roughly 20% if you used test_size=0.2)
comparison_table['Small Split Check (%)'] = (
    small_test_counts / orig_counts) * 100
comparison_table['Large Split Check (%)'] = (
    large_test_counts / orig_counts) * 100

# 5. Clean up the display
print("--- Detailed Distribution Table (class1) ---")
print(comparison_table.round(2))

--- Detailed Distribution Table (class1) ---
                                Original (count)  Original (%)  Small Test Set (count)  Small Test Set (%)  Large Test Set (count)  Large Test Set (%)  Small Split Check (%)  Large Split Check (%)
class1                                                                                                                                                                                              
BruteForce                                 47241          5.76                    12.0                 6.0                   115.0                5.75                   0.03                   0.24
C&C                                         2863          0.35                     1.0                 0.5                     7.0                0.35                   0.03                   0.24
Dictionary                                  2572          0.31                     1.0                 0.5                     6.0                0.30                 

In [5]:
from tabgan.sampler import GANGenerator, ForestDiffusionGenerator, LLMGenerator

# , 'class1', 'class2', 'class3'
categorical_features = ['Protocol', 'Service']

# new_train1, new_target1 = OriginalGenerator(
# ).generate_data_pipe(train, target, test)
# new_train2, new_target2 = GANGenerator(gen_params={
#                                        "batch_size": 500, "epochs": 10, "patience": 5}).generate_data_pipe(train, target, test)
# new_train3, new_target3 = ForestDiffusionGenerator(
# ).generate_data_pipe(train, target, test)
# new_train4, new_target4 = LLMGenerator(gen_params={
#                                        "batch_size": 32, "epochs": 4, "llm": "distilgpt2", "max_length": 500}).generate_data_pipe(train, target, test)


small_GAN_train, small_GAN_target = GANGenerator(gen_params={
    'cat_cols': categorical_features, "deep_copy": True, "only_generated_data": True,
    "batch_size": 200, "patience": 25, "epochs": 100}) \
    .generate_data_pipe(train_df=X_train_small, target=y_train_small, test_df=X_test_small)

large_GAN_train, large_GAN_target = GANGenerator(gen_params={
    'cat_cols': categorical_features, "deep_copy": True, "only_generated_data": True,
    "batch_size": 500, "patience": 25, "epochs": 500}) \
    .generate_data_pipe(train_df=X_train_large, target=y_train_large, test_df=X_test_large)

c:\Users\antho\.conda\envs\CS9840\Lib\site-packages\tabgan\__init__.py:2: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution
Fitting CTGAN transformers for each column:   0%|          | 0/66 [00:00<?, ?it/s]


ValueError: could not convert string to float: '12/02/2020'